In [0]:
import json
import logging
import concurrent.futures
from pyspark.sql.functions import current_timestamp, lit, regexp_extract, col

# Configuración del logger
logger = logging.getLogger("IngestionPipeline")
logger.setLevel(logging.INFO)

catalog_name = "real_state_project"
spark.catalog.setCurrentCatalog(catalog_name)

base_landing_path = f"/Volumes/{catalog_name}/raw_data_real_state/real_state_csv_landing_volume/"

def procesar_portal(row):
    row_dict = row.asDict()
    scraper_tool     = row_dict.get("scraper_tool")
    portal           = row_dict.get("portal")
    property_type    = row_dict.get("property_type")
    operation_type   = row_dict.get("operation_type")
    file_prefix      = row_dict.get("file_prefix")
    schema_and_table = row_dict.get("target_table")
    
    read_options_str = row_dict.get("read_options", "{}")
    try:
        read_options_dict = json.loads(read_options_str) if read_options_str else {}
    except json.JSONDecodeError:
        read_options_dict = {}

    full_target_table = f"{catalog_name}.{schema_and_table}"
    source_path = f"{base_landing_path}{file_prefix}*"
    
    checkpoint_path = f"{base_landing_path}_checkpoints/{file_prefix}"
    schema_location_path = f"{base_landing_path}_schemas/{file_prefix}"

    # ==========================================================================
    # LECTURA CON NOTIFICACIONES ACTIVADAS
    # ==========================================================================
    try:
        df_raw = spark.readStream \
            .format("cloudFiles") \
            .option("cloudFiles.format", "csv") \
            .option("cloudFiles.rescuedDataColumn", "_rescued_data") \
            .option("cloudFiles.schemaLocation", schema_location_path) \
            .option("cloudFiles.useNotifications", "true") \
            .options(**read_options_dict) \
            .load(source_path) # Auto Loader se encarga de SQS/SNS aquí

        df_enriched = df_raw \
            .withColumn("scraper_tool", lit(scraper_tool)) \
            .withColumn("portal", lit(portal)) \
            .withColumn("property_type", lit(property_type)) \
            .withColumn("operation_type", lit(operation_type)) \
            .withColumn("source_file", col("_metadata.file_path")) \
            .withColumn("extraction_date", regexp_extract("source_file", r"(\d{4}-\d{2}-\d{2})", 1)) \
            .withColumn("ingested_at", current_timestamp())

        query = df_enriched.writeStream \
            .format("delta") \
            .outputMode("append") \
            .option("checkpointLocation", checkpoint_path) \
            .option("mergeSchema", "true") \
            .trigger(availableNow=True) \
            .toTable(schema_and_table)

        query.awaitTermination()
        
        # Extraemos las filas reales procesadas para el reporte
        metrics = query.lastProgress
        num_rows = metrics.get('numInputRows', 0) if metrics else 0
        
        return f"✅ ÉXITO: {full_target_table} procesada. ({num_rows} filas de los eventos de S3)."
        
    except Exception as e:
        error_str = str(e)
        
        # EL NUEVO "ESCUDO": Atrapamos el error de carpeta vacía sin usar dbutils.fs.ls
        if "CF_EMPTY_DIR_FOR_SCHEMA_INFERENCE" in error_str:
            return f"⚠️ SKIP: {file_prefix} (Aún no hay archivos depositados para inferir esquema)."
            
        # Si es cualquier otro error (como el 403 de AWS o un CSV corrupto), lo marcamos como fallo
        return f"💥 ERROR: Falló la ingesta para {file_prefix}. Detalle: {error_str}"

# ==============================================================================
# EJECUCIÓN MULTIPROCESAMIENTO
# ==============================================================================
df_control = spark.table(f"{catalog_name}.raw_data_real_state.ingestion_map") \
                  .filter(col("is_active") == True)

MAX_HILOS_CONCURRENTES = 10 
print("-" * 60)
print(f"Lanzando {MAX_HILOS_CONCURRENTES} hilos con Event Notifications (SQS/SNS)...")

with concurrent.futures.ThreadPoolExecutor(max_workers=MAX_HILOS_CONCURRENTES) as executor:
    # Usamos submit para manejar el futuro de manera más segura si hay fallos a nivel hilo
    futuros = {executor.submit(procesar_portal, row): row for row in df_control.collect()}
    
    resultados = []
    for futuro in concurrent.futures.as_completed(futuros):
        try:
            resultados.append(futuro.result())
        except Exception as exc:
            # Captura fallos que rompen el worker por completo
            resultados.append(f"💥 ERROR FATAL en el hilo: {exc}")

print("\n" + "=" * 60)
print("REPORTE FINAL DE INGESTA")
print("=" * 60)
for reporte in resultados:
    print(reporte)